[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/082.%20The%20Single%20Facility%20Location%20Problem%20%28Center%20of%20Gravity%29/P82-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 82. The Single Facility Location Problem
## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

### Purpose and scope
This notebook applies the center of gravity method to estimate a continuous location for one distribution center serving a set of demand locations. It calculates the demand-weighted centroid, evaluates the resulting location using demand-weighted Euclidean distance, visualizes the network, and examines how the location changes under specified demand scenarios.

The center of gravity method has an important mathematical interpretation: the weighted centroid exactly minimizes the **sum of weighted squared Euclidean distances**. The notebook also reports the **sum of weighted Euclidean distances** as a separate distance-based performance measure. These are different objectives; the centroid is not, in general, the exact minimizer of the latter.

### Model assumptions
- The facility may be placed anywhere in an unconstrained, continuous two-dimensional plane.
- Each demand location has a nonnegative demand weight, and the total demand is positive.
- Euclidean distance represents straight-line separation in the coordinate system. Road networks, terrain, travel time, and location-dependent transportation rates are not modeled.
- The center of gravity objective is the sum of demand-weighted **squared** Euclidean distances. The reported weighted Euclidean distance is an additional evaluation measure, not the objective minimized by the centroid formula.
- The sensitivity scenarios change demands or add a location while keeping the method and its assumptions unchanged.

### Method and learning objectives
For each demand location, its demand determines how strongly it influences the weighted average. The calculation proceeds by defining coordinates and demands, finding the weighted average of each coordinate, evaluating distances from the resulting location, and then examining specified changes in the input data. The arithmetic requires a number of operations proportional to the number of locations, so its running time is $O(n)$ for $n$ demand locations.

The weighted centroid is exact for the squared-distance objective because differentiating that objective with respect to the facility coordinates and setting both derivatives to zero gives the weighted-average formulas. If transportation cost is instead assumed to be directly proportional to ordinary Euclidean distance, the centroid remains a useful center-of-gravity estimate, but it does not generally provide an exact optimum for that different objective.

### Example: MegaCorp's distribution network
MegaCorp operates five retail stores. Their coordinates describe positions in a common coordinate system, and their weekly demands are the location weights:

| Store | Coordinates $(x_i, y_i)$ | Weekly demand $w_i$ |
|:--|:--:|--:|
| 1 | $(2, 3)$ | 150 units |
| 2 | $(8, 1)$ | 200 units |
| 3 | $(6, 7)$ | 180 units |
| 4 | $(12, 4)$ | 160 units |
| 5 | $(4, 9)$ | 110 units |

The goal is to calculate the center of gravity for the facility and assess the resulting distances. Coordinates are treated as being in a consistent distance unit; demand-weighted distance therefore has units of demand times distance.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A consistent plotting style makes comparisons across the figures easier to read.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class DemandLocation:
    """A demand point with planar coordinates and a nonnegative weight."""

    id: int
    x: float
    y: float
    demand: float

    def distance_to(self, x: float, y: float) -> float:
        """Return the Euclidean distance from this location to (x, y)."""
        return float(np.hypot(self.x - x, self.y - y))


# The same input locations are used in the centroid calculation and all analyses.
stores = [
    DemandLocation(id=1, x=2, y=3, demand=150),
    DemandLocation(id=2, x=8, y=1, demand=200),
    DemandLocation(id=3, x=6, y=7, demand=180),
    DemandLocation(id=4, x=12, y=4, demand=160),
    DemandLocation(id=5, x=4, y=9, demand=110),
]

print("MegaCorp store locations:")
for store in stores:
    print(
        f"Store {store.id}: ({store.x}, {store.y}), "
        f"weekly demand {store.demand} units"
    )

### The center of gravity calculation
Let $I$ be the set of demand locations, and let $w_i > 0$ and $(x_i, y_i)$ denote the demand and coordinates of location $i$. For a candidate facility position $(x, y)$, the squared-distance objective is

$$
F(x,y) = \sum_{i \in I} w_i\left[(x-x_i)^2 + (y-y_i)^2\right].
$$

Each term is the squared Euclidean distance to a demand point, weighted by that point's demand. Setting the partial derivatives of $F$ to zero gives the unique minimizer when total demand is positive:

$$
x^* = \frac{\sum_{i \in I} w_i x_i}{\sum_{i \in I} w_i},
\qquad
y^* = \frac{\sum_{i \in I} w_i y_i}{\sum_{i \in I} w_i}.
$$

Here, $I$ is the set of stores, $w_i$ is the demand at store $i$, $(x_i,y_i)$ is its coordinate, and $(x^*,y^*)$ is the center of gravity. In the code, `calculate_center_of_gravity` computes the weighted sums and total demand directly. A zero or negative demand weight is excluded by the model assumptions; the function checks that the input is nonempty, all demands are nonnegative, and total demand is positive.

In [ ]:
def calculate_center_of_gravity(
    locations: List[DemandLocation],
) -> Tuple[float, float, float]:
    """Return the weighted centroid coordinates and total demand.

    The weighted centroid minimizes the sum of demand-weighted squared
    Euclidean distances over an unconstrained continuous plane.
    """
    if not locations:
        raise ValueError("At least one demand location is required.")
    if any(location.demand < 0 for location in locations):
        raise ValueError("Demand weights must be nonnegative.")

    total_demand = sum(location.demand for location in locations)
    if total_demand <= 0:
        raise ValueError("Total demand must be positive.")

    # Dividing each weighted coordinate sum by total demand gives the
    # coordinate-wise weighted averages in the mathematical formulation.
    weighted_x = sum(location.x * location.demand for location in locations)
    weighted_y = sum(location.y * location.demand for location in locations)
    optimal_x = weighted_x / total_demand
    optimal_y = weighted_y / total_demand

    return optimal_x, optimal_y, total_demand


optimal_x, optimal_y, total_demand = calculate_center_of_gravity(stores)

print(f"Total weekly demand: {total_demand:g} units")
print(f"Center of gravity: ({optimal_x:.3f}, {optimal_y:.3f})")

### Evaluating weighted Euclidean distance
After finding the centroid, we can calculate the ordinary Euclidean distance from the facility to each store:

$$
d_i = \sqrt{(x^*-x_i)^2 + (y^*-y_i)^2}.
$$

The notebook reports the sum of these distances weighted by demand,

$$
D = \sum_{i \in I} w_i d_i,
$$

and the average distance per demand unit, $D / \sum_{i \in I} w_i$. These measures help interpret the centroid's geographic service distances. They are not the squared-distance objective $F(x,y)$ minimized by the center of gravity formula. In the code, `calculate_total_weighted_distance` computes each $d_i$, its weighted contribution $w_i d_i$, and their total.

In [ ]:
def calculate_total_weighted_distance(
    locations: List[DemandLocation],
    facility_x: float,
    facility_y: float,
    verbose: bool = True,
) -> Tuple[float, List[Dict[str, object]]]:
    """Calculate demand-weighted Euclidean distance and store-level details."""
    if not locations:
        raise ValueError("At least one demand location is required.")
    if any(location.demand < 0 for location in locations):
        raise ValueError("Demand weights must be nonnegative.")

    total_weighted_distance = 0.0
    distance_details: List[Dict[str, object]] = []

    for store in locations:
        distance = store.distance_to(facility_x, facility_y)
        weighted_cost = store.demand * distance
        total_weighted_distance += weighted_cost
        distance_details.append(
            {
                "store": store.id,
                "coords": (store.x, store.y),
                "demand": store.demand,
                "distance": distance,
                "weighted_cost": weighted_cost,
            }
        )

    if verbose:
        print("\nWeighted Euclidean distance by store:")
        print("-" * 64)
        print(
            f"{'Store':<8}{'Coordinates':<16}{'Demand':>10}"
            f"{'Distance':>12}{'Weighted distance':>18}"
        )
        for detail in distance_details:
            x_coord, y_coord = detail["coords"]
            print(
                f"{detail['store']:<8}"
                f"({x_coord:g}, {y_coord:g})"
                f"{'':<7}{detail['demand']:>10g}"
                f"{detail['distance']:>12.2f}"
                f"{detail['weighted_cost']:>18.2f}"
            )
        print("-" * 64)
        print(f"Total weighted distance: {total_weighted_distance:.2f}")

    return total_weighted_distance, distance_details


total_cost, cost_breakdown = calculate_total_weighted_distance(
    stores, optimal_x, optimal_y
)
print(f"Average distance per demand unit: {total_cost / total_demand:.3f}")

### Visualizing locations and distances
The four panels below show the demand points and centroid, the demand distribution, each store's contribution to weighted distance, and the relationship between demand and distance. Marker area in the location map is proportional to demand. The dashed lines connect the centroid to the stores; they show straight-line distances rather than routes on a road network.

In [ ]:
def visualize_facility_location(
    locations: List[DemandLocation],
    facility_x: float,
    facility_y: float,
    distance_details: List[Dict[str, object]],
) -> None:
    """Plot demand locations, demand levels, and distance contributions."""
    if len(locations) != len(distance_details):
        raise ValueError("Distance details must match the demand locations.")

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 12)
    )
    fig.suptitle(
        "Single-Facility Location: Center of Gravity",
        fontsize=16,
        fontweight="bold",
    )

    # The map links each demand point to the calculated facility position.
    ax1.set_title("Demand Locations and Facility", fontweight="bold")
    for store in locations:
        ax1.scatter(
            store.x,
            store.y,
            s=store.demand * 1.5,
            alpha=0.7,
            label=f"Store {store.id}",
            edgecolors="black",
            linewidth=1,
        )
        ax1.annotate(
            f"Store {store.id}\n({store.x:g}, {store.y:g})\n"
            f"{store.demand:g} units",
            xy=(store.x, store.y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.8},
        )
        ax1.plot(
            [facility_x, store.x],
            [facility_y, store.y],
            "k--",
            alpha=0.35,
            linewidth=1,
        )

    ax1.scatter(
        facility_x,
        facility_y,
        s=300,
        c="red",
        marker="*",
        label="Center of gravity",
        edgecolors="black",
        linewidth=1,
        zorder=3,
    )
    ax1.annotate(
        f"Facility ({facility_x:.3f}, {facility_y:.3f})",
        xy=(facility_x, facility_y),
        xytext=(10, 10),
        textcoords="offset points",
        fontsize=9,
        fontweight="bold",
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "mistyrose", "alpha": 0.9},
    )
    ax1.set_xlabel("X coordinate (distance units)")
    ax1.set_ylabel("Y coordinate (distance units)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="best", fontsize=8)

    # A bar chart makes the input weights easy to compare.
    store_ids = [store.id for store in locations]
    demands = [store.demand for store in locations]
    colors = plt.cm.Set3(np.linspace(0, 1, len(locations)))
    ax2.set_title("Weekly Demand by Store", fontweight="bold")
    bars = ax2.bar(store_ids, demands, color=colors, edgecolor="black", alpha=0.8)
    ax2.set_xlabel("Store ID")
    ax2.set_ylabel("Weekly demand (units)")
    ax2.grid(True, axis="y", alpha=0.3)
    for bar, demand in zip(bars, demands):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{demand:g}",
            ha="center",
            va="bottom",
        )

    # These bars decompose the reported weighted-distance measure by store.
    weighted_costs = [detail["weighted_cost"] for detail in distance_details]
    store_names = [f"Store {store.id}" for store in locations]
    ax3.set_title("Weighted Distance Contribution", fontweight="bold")
    bars = ax3.bar(
        store_names,
        weighted_costs,
        color=colors,
        edgecolor="black",
        alpha=0.8,
    )
    ax3.set_xlabel("Demand location")
    ax3.set_ylabel("Demand-weighted distance (units × distance)")
    ax3.tick_params(axis="x", rotation=35)
    ax3.grid(True, axis="y", alpha=0.3)
    for bar, contribution in zip(bars, weighted_costs):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{contribution:.0f}",
            ha="center",
            va="bottom",
        )

    # The scatter plot describes this example; it does not imply causation.
    distances = [detail["distance"] for detail in distance_details]
    ax4.set_title("Demand and Distance to Facility", fontweight="bold")
    ax4.scatter(
        demands,
        distances,
        s=90,
        c=colors,
        edgecolors="black",
        alpha=0.8,
    )
    for store, demand, distance in zip(locations, demands, distances):
        ax4.annotate(
            f"Store {store.id}",
            (demand, distance),
            xytext=(5, 5),
            textcoords="offset points",
        )
    ax4.set_xlabel("Weekly demand (units)")
    ax4.set_ylabel("Euclidean distance (distance units)")
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()


visualize_facility_location(stores, optimal_x, optimal_y, cost_breakdown)

### Sensitivity analysis
A sensitivity analysis changes selected inputs and recomputes the center of gravity and weighted Euclidean distance for each scenario. This shows how demand weights and the set of demand locations influence the result. Each scenario is evaluated at its own calculated centroid. The reported percentage change compares its weighted Euclidean distance with the base scenario's weighted Euclidean distance; it is not a comparison of the squared-distance objective.

In [ ]:
def perform_sensitivity_analysis(
    locations: List[DemandLocation],
) -> Dict[str, object]:
    """Recompute the center of gravity for the specified demand scenarios."""
    if len(locations) < 4:
        raise ValueError("The sensitivity scenarios require at least four locations.")

    print("\n" + "=" * 72)
    print("SENSITIVITY ANALYSIS")
    print("=" * 72)

    scenarios_to_evaluate = [
        (
            "Base case",
            locations.copy(),
            "Original demands",
        ),
        (
            "Store 1 +50%",
            [
                DemandLocation(
                    id=location.id,
                    x=location.x,
                    y=location.y,
                    demand=location.demand * 1.5 if index == 0 else location.demand,
                )
                for index, location in enumerate(locations)
            ],
            "Store 1 demand increased by 50%",
        ),
        (
            "Store 4 -30%",
            [
                DemandLocation(
                    id=location.id,
                    x=location.x,
                    y=location.y,
                    demand=location.demand * 0.7 if index == 3 else location.demand,
                )
                for index, location in enumerate(locations)
            ],
            "Store 4 demand decreased by 30%",
        ),
        (
            "Add Store 6",
            locations + [DemandLocation(id=6, x=10, y=8, demand=250)],
            "Add a store at (10, 8) with demand 250 units",
        ),
        (
            "Equal demands",
            [
                DemandLocation(
                    id=location.id,
                    x=location.x,
                    y=location.y,
                    demand=sum(item.demand for item in locations) / len(locations),
                )
                for location in locations
            ],
            "Set each store's demand to the base-case average",
        ),
    ]

    scenario_results: List[Dict[str, object]] = []
    for name, scenario_locations, description in scenarios_to_evaluate:
        scenario_x, scenario_y, scenario_demand = calculate_center_of_gravity(
            scenario_locations
        )
        scenario_cost, _ = calculate_total_weighted_distance(
            scenario_locations,
            scenario_x,
            scenario_y,
            verbose=False,
        )
        scenario_results.append(
            {
                "name": name,
                "x": scenario_x,
                "y": scenario_y,
                "cost": scenario_cost,
                "total_demand": scenario_demand,
                "description": description,
                "locations": scenario_locations,
            }
        )

    base_cost = scenario_results[0]["cost"]
    print(
        f"{'Scenario':<16}{'Location':<22}{'Weighted distance':>20}"
        f"{'Change from base':>20}"
    )
    print("-" * 78)
    for result in scenario_results:
        percentage_change = (result["cost"] - base_cost) / base_cost * 100
        location_text = f"({result['x']:.3f}, {result['y']:.3f})"
        cost_text = f"{result['cost']:.2f}"
        change_text = f"{percentage_change:+.1f}%"
        print(
            f"{result['name']:<16}{location_text:<22}"
            f"{cost_text:>20}{change_text:>20}"
        )

    return {"base_cost": base_cost, "scenarios": scenario_results}


sensitivity_results = perform_sensitivity_analysis(stores)

### Summary dashboard
The final dashboard brings together the base-case result and the sensitivity scenarios. The comparison panels use the scenario records returned by `perform_sensitivity_analysis`, so the plotted locations and costs correspond to the same calculations shown in the table. The cost-change colors distinguish increases from decreases relative to the base case.

In [ ]:
def create_summary_dashboard(
    locations: List[DemandLocation],
    facility_x: float,
    facility_y: float,
    total_demand: float,
    total_weighted_distance: float,
    sensitivity_results: Dict[str, object],
) -> None:
    """Plot base-case results and the computed sensitivity scenarios."""
    scenarios = sensitivity_results["scenarios"]
    base_cost = sensitivity_results["base_cost"]

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(16, 12)
    )
    fig.suptitle(
        "Center of Gravity Method: Summary Dashboard",
        fontsize=16,
        fontweight="bold",
    )

    # The text panel states the result and distinguishes the two distance measures.
    ax1.axis("off")
    results_text = (
        "BASE-CASE RESULTS\n"
        "=================\n\n"
        f"Center of gravity: ({facility_x:.3f}, {facility_y:.3f})\n"
        f"Total weekly demand: {total_demand:g} units\n"
        f"Weighted Euclidean distance: {total_weighted_distance:.2f}\n"
        f"Average distance per demand unit: "
        f"{total_weighted_distance / total_demand:.3f}\n\n"
        "The centroid minimizes weighted squared\n"
        "Euclidean distance. The weighted Euclidean\n"
        "distance shown here is a separate measure."
    )
    ax1.text(
        0.05,
        0.95,
        results_text,
        transform=ax1.transAxes,
        fontsize=11,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round,pad=0.6", "facecolor": "lightblue", "alpha": 0.8},
    )

    # Scenario coordinates come directly from the calculated scenario records.
    scenario_names = [result["name"] for result in scenarios]
    x_coords = [result["x"] for result in scenarios]
    y_coords = [result["y"] for result in scenarios]
    ax2.set_title("Center of Gravity by Scenario", fontweight="bold")
    scatter = ax2.scatter(
        x_coords,
        y_coords,
        c=range(len(scenarios)),
        cmap="viridis",
        s=100,
        edgecolors="black",
    )
    for x_coord, y_coord, name in zip(x_coords, y_coords, scenario_names):
        ax2.annotate(
            name,
            (x_coord, y_coord),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )
    for index in range(1, len(scenarios)):
        ax2.annotate(
            "",
            xy=(x_coords[index], y_coords[index]),
            xytext=(x_coords[0], y_coords[0]),
            arrowprops={"arrowstyle": "->", "color": "gray", "alpha": 0.5},
        )
    ax2.set_xlabel("X coordinate (distance units)")
    ax2.set_ylabel("Y coordinate (distance units)")
    ax2.grid(True, alpha=0.3)
    colorbar = fig.colorbar(scatter, ax=ax2, ticks=range(len(scenarios)))
    colorbar.set_label("Scenario index")
    colorbar.set_ticklabels([str(index) for index in range(len(scenarios))])

    # Bar heights and labels use the same weighted-distance measure as the table.
    costs = [result["cost"] for result in scenarios]
    cost_changes = [(cost - base_cost) / base_cost * 100 for cost in costs]
    bar_colors = ["green" if change <= 0 else "red" for change in cost_changes]
    ax3.set_title("Weighted Euclidean Distance by Scenario", fontweight="bold")
    bars = ax3.bar(
        scenario_names,
        costs,
        color=bar_colors,
        alpha=0.75,
        edgecolor="black",
    )
    for bar, change in zip(bars, cost_changes):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{change:+.1f}%",
            ha="center",
            va="bottom",
            fontsize=8,
        )
    ax3.set_ylabel("Demand-weighted distance (units × distance)")
    ax3.set_xlabel("Scenario")
    ax3.tick_params(axis="x", rotation=30)
    ax3.grid(True, axis="y", alpha=0.3)
    ax3.plot([], [], color="green", linewidth=8, label="At or below base")
    ax3.plot([], [], color="red", linewidth=8, label="Above base")
    ax3.legend()

    # The final panel summarizes the model's interpretation and limits.
    ax4.axis("off")
    insights_text = (
        "METHOD INTERPRETATION\n"
        "=====================\n\n"
        "MATHEMATICAL RESULT\n"
        "The demand-weighted centroid minimizes\n"
        "weighted squared Euclidean distance in\n"
        "an unconstrained continuous plane.\n\n"
        "COMPUTATION\n"
        "Weighted coordinate sums require O(n)\n"
        "work for n demand locations.\n\n"
        "INTERPRETATION\n"
        "Greater demand gives a location greater\n"
        "influence on the weighted average.\n\n"
        "SCOPE LIMITS\n"
        "The model does not represent roads,\n"
        "restricted sites, or other constraints.\n"
        "The centroid is not generally the exact\n"
        "minimizer of weighted Euclidean distance."
    )
    ax4.text(
        0.05,
        0.95,
        insights_text,
        transform=ax4.transAxes,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round,pad=0.6", "facecolor": "lightgreen", "alpha": 0.8},
    )

    fig.tight_layout()
    plt.show()


create_summary_dashboard(
    stores,
    optimal_x,
    optimal_y,
    total_demand,
    total_cost,
    sensitivity_results,
)

## Strengths, limitations, and appropriate use

### Strengths
- The weighted centroid has a direct mathematical derivation and is the exact minimizer of the stated weighted squared Euclidean-distance objective.
- Its calculation is simple and takes time proportional to the number of demand locations.
- Demand-weighted coordinate averages are straightforward to interpret and provide a useful initial estimate for strategic planning.
- The method can be applied to any finite number of demand locations, provided the total demand is positive.

### Limitations
- The model assumes a continuous, unconstrained feasible plane and may place the facility where construction or operation is infeasible.
- It does not account for roads, travel time, terrain, or location-specific costs.
- Its exact optimality guarantee applies to weighted squared Euclidean distance, not generally to weighted Euclidean distance or a road-network transportation cost.
- It considers one objective and does not represent capacity, service, or other operational constraints.

### When to use this method
The center of gravity method is appropriate as a transparent, fast estimate when a continuous-plane model is a useful first approximation. It is also useful for understanding how demand weights influence a location and for establishing a baseline for further analysis. When feasible sites are discrete, geographic restrictions are important, or travel costs differ substantially from straight-line distance, those features must be represented in a more detailed model.

### Key takeaway
The center of gravity is a demand-weighted average of coordinates. Its mathematical guarantee is specific: it minimizes the sum of demand-weighted squared Euclidean distances under the stated continuous-space assumptions. The notebook's weighted Euclidean-distance totals provide a separate, interpretable measure of service distance at that calculated location.